# Getting an overview of the EMISSOR scenarios

This notebook shows how you get an overview of the scenarios in your EMISSOR folder.

The notebook requires the ```emissor``` package that you can install through pip.

In [40]:
#!pip install emissor

In [10]:
from datetime import datetime
import os
from emissor.persistence import ScenarioStorage
from emissor.persistence.persistence import ScenarioController
from emissor.processing.api import SignalProcessor
from emissor.representation.scenario import Modality, Signal, TextSignal

## Load the different scenarios from the EMISSOR folder

In [11]:
emissor = "./emissor"
scenario = []
# Get list of scenarios from the emissor path
try:
    scenarios = [f for f in os.listdir(emissor) if os.path.isdir(os.path.join(emissor, f))]
    print(f'There are %s scenarios are:', len(scenarios))
except FileNotFoundError:
    print(f"Error: The path '{emissor}' does not exist.")
except Exception as e:
    print(f"An error occurred: {str(e)}")
        

There are %s scenarios are: 2


## Getting meta data and sginals from a scenario

The next functions 1) read the meta data from the scenario JSON file and the signals from the text and image JSON files. 

In [12]:
def get_date_duration_in_minutes(scenario_ctrl):
    start = 0
    end = 0
    duration = 0
    date = None
    try:
        start = int(scenario_ctrl.scenario.start)
        end = int(scenario_ctrl.scenario.end)
        date = datetime.fromtimestamp(start/1000).strftime('%Y-%m-%d')
    except:
        print(f'\tError getting duration: start={scenario_ctrl.scenario.start}, end={scenario_ctrl.scenario.end}')
    if start>0 and end>0:
        duration = (end - start) / 60000
    return date, duration
    
def get_meta_data (emissor_folder:str, scenario_id:str):
    scenario_folder = os.path.join(emissor_folder, scenario_id)
    scenario_storage = ScenarioStorage(emissor_folder)
    scenario_ctrl = scenario_storage.load_scenario(scenario_id)
    speaker = "No speaker"
    agent = "No agent"
    location = "No location"
    people = "Not in context"
    objects = "Not in context"
    duration = 0
    try:
        speaker = scenario_ctrl.scenario.context.speaker["name"] if "name" in scenario_ctrl.scenario.context.speaker else "No speaker"
    except:
        print("\tNo speaker in context")
    try:
        agent = scenario_ctrl.scenario.context.agent["name"] if "name" in scenario_ctrl.scenario.context.agent else "No agent"
    except:
        print("\tNo speaker in context")
    try:
        location = scenario_ctrl.scenario.context.location_id  #### Change this to location name when this implemented
    except:
        print("\tNo location id in context")
    try:
        people = scenario_ctrl.scenario.context.persons
    except:
        print("\tNo location id in context")
    try:
        objects = scenario_ctrl.scenario.context.objects
    except:
        print("\tNo location id in context")
    date, duration = get_date_duration_in_minutes(scenario_ctrl)
    return speaker, agent, location, people, objects, date, duration

def get_text_signals_from_a_scenario(emissor_folder:str, scenario_id:str):
    text_signals=[]
    scenario_folder = os.path.join(emissor_folder, scenario_id)
    scenario_storage = ScenarioStorage(emissor_folder)
    scenario_ctrl = scenario_storage.load_scenario(scenario_id)
    try:
        text_signals = scenario_ctrl.get_signals(Modality.TEXT)
    except:
        print('Error loading text signals from text.json')
    return text_signals

def get_image_signals_from_a_scenario(emissor_folder:str, scenario_id:str):
    text_signals=[]
    scenario_folder = os.path.join(emissor_folder, scenario_id)
    scenario_storage = ScenarioStorage(emissor_folder)
    scenario_ctrl = scenario_storage.load_scenario(scenario_id)
    try:
        text_signals = scenario_ctrl.get_signals(Modality.IMAGE)
    except:
        print('Error loading image signals from text.json')
    return text_signals

The next loop will get the basic information for each scenario and print the result:

In [13]:
for scenario in scenarios:
    print(f"scenario={scenario}")
    speaker, agent, location, people, objects, date, duration = get_meta_data(emissor, scenario)
    text_signals = get_text_signals_from_a_scenario(emissor, scenario)
    image_signals = get_image_signals_from_a_scenario(emissor, scenario)
    print(f"\tdate={str(date)}, duration in minutes={round(duration, 2)}")
    print(f"\tspeaker={speaker}, agent={agent}, seen people={people}, seen objects={objects}, text signals={len(text_signals)}, image signals={len(image_signals)}")
    print()

scenario=0985799c-80eb-416c-b55e-f11ffc4ac259
	No location id in context
	No location id in context
	No location id in context
	date=2025-10-10, duration in minutes=26.25
	speaker=Piek, agent=Ai2Thor, seen people=Not in context, seen objects=Not in context, text signals=24, image signals=2

scenario=b387db06-934e-405b-9d4e-f7e5c27b440a
	Error getting duration: start=1762517202633, end=None
	date=None, duration in minutes=0
	speaker=Luis, agent=Leolani, seen people=[], seen objects=[], text signals=9, image signals=0



## Getting the conversation from a scenario

The next function gets the speaker from each Text Signal. If the value is ```value='SPEAKER'``` it is the speaker that is set in the scenario meta data.

In [14]:
def get_speaker_from_text_signal(textSignal: TextSignal):
    speaker = None
    mentions = textSignal.mentions
    for mention in mentions:
        annotations = mention.annotations
        for annotation in annotations:
            if annotation.type == 'ConversationalAgent':
                speaker = annotation.value
                break
        if speaker:
            break
    return speaker

In [16]:
scenario = "b387db06-934e-405b-9d4e-f7e5c27b440a"
speaker, agent, _, _, _, _, _ = get_meta_data(emissor, scenario)
# print(speaker, agent)
text_signals = get_text_signals_from_a_scenario(emissor, scenario)
for text_signal in text_signals:
    signal_speaker = ""
    signal_speaker = get_speaker_from_text_signal(text_signal)
    if signal_speaker=='SPEAKER':
        signal_speaker = speaker
    else:
        signal_speaker = agent
    print(signal_speaker, ":", text_signal.text)

	Error getting duration: start=1762517202633, end=None
Leolani : Yo Do you want to talk to me Luis?
Luis : Yes
Leolani : I have nothing more to say.
Luis : I live in Amstelveen.
Leolani : Has Luis visited the agent?
Luis : I visited Piek.
Luis : Do you know Piek?
Leolani : Ik wil het weten. Heeft Luis muzikale werken?
Leolani : Someone said that I know Piek.


In [17]:
scenario = "0985799c-80eb-416c-b55e-f11ffc4ac259"
speaker, agent, _, _, _, _, _ = get_meta_data(emissor, scenario)
# print(speaker, agent)
text_signals = get_text_signals_from_a_scenario(emissor, scenario)
for text_signal in text_signals:
    signal_speaker = ""
    signal_speaker = get_speaker_from_text_signal(text_signal)
    if signal_speaker=='SPEAKER':
        signal_speaker = speaker
    else:
        signal_speaker = agent
    print(signal_speaker, ":", text_signal.text)

	No location id in context
	No location id in context
	No location id in context
Ai2Thor : Hi Piek. Tell me what to do.
Ai2Thor : This is what I can do:('I can do the following:', "['find', 'describe', 'move', 'go', 'turn', 'forward', 'back', 'left', 'right', 'open', 'close', 'look']")
Ai2Thor : forward
Ai2Thor : move forward
Ai2Thor : MoveAhead
Ai2Thor : what do you see.
Ai2Thor : Sorry I do not get that:what
Ai2Thor : look
Ai2Thor : describe
Ai2Thor : I see 67 things there.
Apple
Bottle
	I can break it.
Bowl
Bread
ButterKnife
Cabinet
	I can open it.
Cabinet
	I can open it.
Cabinet
	I can open it.
Cabinet
	I can open it.
Cabinet
	I can open it.
Cabinet
	I can open it.
CellPhone
	I can break it.
Chair
	I can move it.
Chair
	I can move it.
CoffeeMachine
	I can move it.
CounterTop
CounterTop
CounterTop
CreditCard
Cup
	I can break it.
DishSponge
Drawer
	I can open it.
Drawer
	I can open it.
Drawer
	I can open it.
Egg
	I can break it.
Faucet
Floor
Fork
Fridge
	I can open it.
GarbageCan
	I 

## End of notebook